# Modelo de suspeita: treino no histórico de 2025

**Problema de negócio.** Num evento climático chegam milhares de avisos em poucos dias. A seguradora quer pagar
rápido os casos simples (fast track) sem deixar passar pagamento indevido (*leakage*). O `score_suspeita`
(0 a 1) faz duas coisas:
- **bloqueia o fast track** quando o score passa de 0,2 (spec 6.3);
- **prioriza a fila do regulador.**

Cada score acima do limiar traz os fatores que mais pesaram, em português, para o regulador e para o parecer.

**Dados.**
- `silver.historico_enriquecido`: 20 mil sinistros regulados em 2025, com as **mesmas features** do evento,
  calculadas pelas mesmas regras do pipeline.
- Rótulo: `restrito/gabarito/gabarito_historico_2025.csv`. Alvo `suspeito` = padrão diferente de `legitimo_*`.

**Validação.**
- Split temporal: treino de jan a set e teste de out a dez/2025.
- Checagem fora do tempo nos lotes de validação do evento de 2026 (parâmetro `lotes_validacao`, padrão: lote 1).
- Os demais lotes são o teste final, na fase 5.

**Decisões (D-021):**
- o clima bruto fica fora do modelo (no histórico ele se confunde com a fonte da evidência, porque não há
  Defesa Civil em 2025); uma ablação mostra o efeito;
- SHAP pelo próprio LightGBM (TreeSHAP exato);
- participação do prestador point-in-time e suavizada.

**Leitura honesta das métricas.**
- Os padrões sintéticos foram plantados com sinais claros. Cinco dos sete padrões suspeitos saem de regras simples
  (sem cobertura, evidência contrária, apólice com menos de 30 dias, laudo com dano anterior e preço ≥ 30% acima
  da tabela). Por isso o notebook mede também essa **linha de base de regras** no mesmo teste.
- O ganho do modelo está no que regra fixa não pega (prestador concentrado) e no **ranking** para a fila.
- No lote 1, só os orçamentos inflados e os danos preexistentes dependem do OCR e da extração por IA; a evidência
  climática vem de joins estruturados.
- Com dados reais, o desempenho seria menor, e a calibração viria da PoC.

In [ ]:
import json
import tempfile
from importlib.metadata import version as versao_pacote
from pathlib import Path

import lightgbm as lgb
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from pyspark.sql import functions as F
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_curve

import ml
import regras
from ml.treino import (CORTE_TESTE, FRACAO_TOPO, LIMIAR_FAST_TRACK, PARAMS_LGBM, PISO_AUC, TETO_SUSPEITOS_ABAIXO_LIMIAR,
                       assinatura, contribuicoes, decidir_promocao, dividir_temporal, metricas, metricas_fast_track,
                       passa_piso, por_padrao, regra_base, treinar)
from regras.features import (COLUNAS_CLIMA_BRUTO, COLUNAS_ENTRADA, COLUNAS_PROIBIDAS, FEATURES, ROTULOS,
                             VERSAO_CONTRATO_FEATURES, montar_features, preparar_entrada, rotulo_suspeito)

dbutils.widgets.text("lotes_validacao", "1")  # noqa: F821 (dbutils é injetado pelo Databricks)
for nome in ("catalogo", "schema_silver", "schema_ml", "experimento"):
    dbutils.widgets.text(nome, "")  # noqa: F821
catalogo, schema_silver, schema_ml, experimento = (dbutils.widgets.get(n) for n in  # noqa: F821
                                                   ("catalogo", "schema_silver", "schema_ml", "experimento"))
LOTES_VALIDACAO = sorted({int(x) for x in dbutils.widgets.get("lotes_validacao").split(",") if x.strip()})  # noqa: F821
assert all((catalogo, schema_silver, schema_ml, experimento)) and LOTES_VALIDACAO, "parâmetros do job ausentes"
SILVER, ML = f"{catalogo}.{schema_silver}", f"{catalogo}.{schema_ml}"
NOME_MODELO = f"{ML}.modelo_suspeita"
TABELA_FEATURES = f"{ML}.features_treino"
GABARITO = f"/Volumes/{catalogo}/{schema_ml}/restrito/gabarito"
VERSOES = {p: versao_pacote(p) for p in ("mlflow", "lightgbm", "scikit-learn", "matplotlib", "pandas", "numpy")}
print(f"modelo: {NOME_MODELO} | features: {TABELA_FEATURES} | experimento: {experimento} | "
      f"lotes de validação: {LOTES_VALIDACAO}")
print("versões:", VERSOES)

modelo: fe_bar_sinistros_catalog.ml.modelo_suspeita | features: fe_bar_sinistros_catalog.ml.features_treino | experimento: /Shared/sinistros_clima/dev/modelo_suspeita | lotes de validação: [1]
versões: {'mlflow': '3.18.0', 'lightgbm': '4.7.0', 'scikit-learn': '1.6.1', 'matplotlib': '3.10.0', 'pandas': '2.2.3', 'numpy': '2.1.3'}


## 1. Dados de treino: histórico enriquecido + rótulo do gabarito restrito

Duas checagens de segurança:
- as colunas de entrada existem com o mesmo nome nas duas tabelas silver (treino e evento);
- nenhuma coluna que vaza o rótulo entra nas features.

O resultado vai para `ml.features_treino`, uma tabela Delta versionada, para o `log_input` do MLflow. As linhas
são lidas em ordem fixa (`id_sinistro`) e o LightGBM roda em modo determinístico: a mesma versão Delta dá o mesmo
modelo.

In [ ]:
for tabela in ("historico_enriquecido", "sinistros_enriquecidos"):
    faltando = set(COLUNAS_ENTRADA + COLUNAS_CLIMA_BRUTO) - set(spark.table(f"{SILVER}.{tabela}").columns)  # noqa: F821
    assert not faltando, f"{tabela} sem as colunas {faltando}"
assert not (set(FEATURES) | set(COLUNAS_ENTRADA)) & COLUNAS_PROIBIDAS, "feature proibida (vaza o rótulo)"

hist = spark.table(f"{SILVER}.historico_enriquecido")  # noqa: F821
gab = spark.read.csv(f"{GABARITO}/gabarito_historico_2025.csv", header=True).select("id_sinistro", "padrao")  # noqa: F821
base = (hist.select("id_sinistro", "data_aviso", *COLUNAS_ENTRADA, *COLUNAS_CLIMA_BRUTO)
        .join(gab, "id_sinistro", "left")
        .withColumn("suspeito", (~F.col("padrao").startswith("legitimo_")).cast("int"))
        .withColumn("conjunto", F.when(F.col("data_aviso") < F.lit(CORTE_TESTE).cast("date"), "treino").otherwise("teste")))
base.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABELA_FEATURES)
spark.sql(f"COMMENT ON TABLE {TABELA_FEATURES} IS 'Features de treino do modelo de suspeita (histórico 2025) com rótulo e conjunto (treino/teste).'")  # noqa: F821
VERSAO_FEATURES = int(spark.sql(f"DESCRIBE HISTORY {TABELA_FEATURES} LIMIT 1").first()["version"])  # noqa: F821

df = spark.sql(f"SELECT * FROM {TABELA_FEATURES} VERSION AS OF {VERSAO_FEATURES} ORDER BY id_sinistro").toPandas()  # noqa: F821
assert len(df) == hist.count(), "o join com o gabarito mudou o número de linhas"
assert (rotulo_suspeito(df["padrao"]) == df["suspeito"]).all()  # falha se algum sinistro ficou sem padrão
treino, teste = dividir_temporal(df)
assert set(treino["conjunto"]) == {"treino"} and set(teste["conjunto"]) == {"teste"}
print(f"features_treino versão {VERSAO_FEATURES}: {len(treino)} treino, {len(teste)} teste")
display(df.groupby(["padrao", "conjunto"]).size().unstack("conjunto").reset_index())  # noqa: F821

features_treino versão 5: 13467 treino, 6533 teste


| padrao | teste | treino |
|---|---|---|
| apolice_recente | 136 | 264 |
| dano_preexistente | 137 | 263 |
| legitimo_complexo | 1668 | 3332 |
| legitimo_simples | 3974 | 8026 |
| orcamento_inflado | 271 | 529 |
| prestador_concentrado | 68 | 132 |
| sem_cobertura | 160 | 440 |
| sem_evento_climatico | 119 | 481 |


## 2. Treino (LightGBM), métricas e linha de base de regras

Métricas da spec 6.1: AUC, e precisão e recall nos 10% mais suspeitos. Como a prevalência passa de 10%, o recall
nos 10% do topo tem um teto (`recall_top10_teto`). Métricas de negócio (spec 6.3):
- `suspeitos_abaixo_limiar`: fração dos suspeitos com score < 0,2, que escapariam do bloqueio do fast track;
- `legitimos_abaixo_limiar`: fração dos legítimos que seguem elegíveis ao fast track;
- `ft_*`: o mesmo, só na população que concorre ao fast track (≤ R$ 5 mil, com cobertura e evidência forte).

A **linha de base** é o OU das regras simples. Ela é medida com as mesmas métricas, com score 0 ou 1.

In [ ]:
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experimento)
consulta = f"SELECT * FROM {TABELA_FEATURES} VERSION AS OF {VERSAO_FEATURES} WHERE conjunto = '{{}}' ORDER BY id_sinistro"
datasets = {c: mlflow.data.from_spark(spark.sql(consulta.format(c)), table_name=TABELA_FEATURES,  # noqa: F821
                                      version=str(VERSAO_FEATURES), name=f"features_treino_{c}")
            for c in ("treino", "teste")}

Xtr, Xte = montar_features(treino), montar_features(teste)
with mlflow.start_run(run_name="lightgbm_sem_clima_bruto") as run:
    RUN_ID = run.info.run_id
    mlflow.log_params({**PARAMS_LGBM, "corte_teste": CORTE_TESTE, "limiar_fast_track": LIMIAR_FAST_TRACK,
                       "fracao_topo": FRACAO_TOPO, "features": ",".join(FEATURES), "com_clima_bruto": False,
                       "contrato_features": VERSAO_CONTRATO_FEATURES, "piso_auc": PISO_AUC,
                       "teto_suspeitos_abaixo_limiar": TETO_SUSPEITOS_ABAIXO_LIMIAR,
                       "features_treino_tabela": TABELA_FEATURES, "features_treino_versao": VERSAO_FEATURES,
                       "lotes_validacao": ",".join(map(str, LOTES_VALIDACAO)),
                       "n_treino": len(treino), "n_teste": len(teste),
                       **{f"versao_{p}": v for p, v in VERSOES.items()}})
    for contexto, ds in datasets.items():
        mlflow.log_input(ds, context=contexto)
    mlflow.set_tags({"fase": "3", "decisao": "D-021", "alvo": "suspeito = padrao != legitimo_*"})
    modelo = treinar(Xtr, treino["suspeito"])
    score_tr, score_te = modelo.predict_proba(Xtr)[:, 1], modelo.predict_proba(Xte)[:, 1]
    regra_te = regra_base(teste)
    METRICAS = (metricas(treino["suspeito"], score_tr, "treino_") | metricas(teste["suspeito"], score_te, "teste_")
                | metricas_fast_track(teste, teste["suspeito"], score_te, "teste_"))
    METRICAS_REGRA = (metricas(teste["suspeito"], regra_te, "regra_teste_")
                      | metricas_fast_track(teste, teste["suspeito"], regra_te, "regra_teste_"))
    mlflow.log_metrics(METRICAS | METRICAS_REGRA)
    tabela_padrao_teste = por_padrao(teste["padrao"], score_te).merge(
        por_padrao(teste["padrao"], regra_te)[["padrao", "pct_acima_limiar"]].rename(
            columns={"pct_acima_limiar": "pct_bloqueado_pela_regra"}), on="padrao")
    mlflow.log_table(tabela_padrao_teste, "por_padrao_teste.json")

display(pd.DataFrame([{"metrica": k.removeprefix("teste_"), "modelo": round(v, 4),  # noqa: F821
                       "regra_base": round(METRICAS_REGRA.get(f"regra_{k}", float("nan")), 4)}
                      for k, v in METRICAS.items() if k.startswith("teste_")]))
display(tabela_padrao_teste)  # noqa: F821

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
/local_disk0/.ephemeral_nfs/envs/pythonEnv-76e69edf-0054-4dc3-9b86-5c1808fd4529/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs

| metrica | modelo | regra_base |
|---|---|---|
| auc | 0.997 | 0.96 |
| ap | 0.9918 | 0.9233 |
| precisao_top10 | 1.0 | 0.9893 |
| recall_top10 | 0.734 | 0.7262 |
| recall_top10_teto | 0.734 | 0.734 |
| suspeitos_abaixo_limiar | 0.0247 | 0.0786 |
| legitimos_abaixo_limiar | 0.9904 | 0.9986 |
| n | 6533.0 | 6533.0 |
| n_suspeitos | 891.0 | 891.0 |
| ft_elegiveis | 3028.0 | 3028.0 |
| ft_suspeitos_elegiveis | 94.0 | 94.0 |
| ft_suspeitos_liberados | 14.0 | 31.0 |
| ft_legitimos_liberados_pct | 0.9973 | 1.0 |


| padrao | n | score_mediano | pct_acima_limiar | pct_bloqueado_pela_regra |
|---|---|---|---|---|
| apolice_recente | 136 | 0.9927898854554846 | 96.3 | 100.0 |
| dano_preexistente | 137 | 0.9942907828913446 | 100.0 | 100.0 |
| legitimo_complexo | 1668 | 0.005138111894461032 | 1.7 | 0.1 |
| legitimo_simples | 3974 | 0.0012083864267568465 | 0.7 | 0.2 |
| orcamento_inflado | 271 | 0.9972568815818376 | 100.0 | 100.0 |
| prestador_concentrado | 68 | 0.9451647880851343 | 77.9 | 0.0 |
| sem_cobertura | 160 | 0.9992260823413293 | 100.0 | 100.0 |
| sem_evento_climatico | 119 | 0.9558758743972536 | 98.3 | 98.3 |


## 3. Artefatos: ROC, PR, matriz de confusão no corte do fast track e SHAP

Os valores SHAP vêm do próprio LightGBM (`pred_contrib=True`, TreeSHAP exato, D-021) e estão na escala do logito.

In [ ]:
def _figura_roc(y, s):
    fpr, tpr, _ = roc_curve(y, s)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label=f"AUC = {METRICAS['teste_auc']:.3f}")
    ax.plot([0, 1], [0, 1], "--", color="grey")
    ax.set(xlabel="Taxa de falsos positivos", ylabel="Taxa de verdadeiros positivos", title="Curva ROC (teste out–dez/2025)")
    ax.legend(loc="lower right")
    return fig


def _figura_pr(y, s):
    precisao, recall, _ = precision_recall_curve(y, s)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall, precisao, label=f"AP = {METRICAS['teste_ap']:.3f}")
    ax.set(xlabel="Recall", ylabel="Precisão", title="Curva precisão × recall (teste)")
    ax.legend(loc="lower left")
    return fig


def _figura_confusao(y, s):
    m = confusion_matrix(y, (s >= LIMIAR_FAST_TRACK).astype(int))
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(m, cmap="Blues")
    for (i, j), v in np.ndenumerate(m):
        ax.text(j, i, str(v), ha="center", va="center")
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["score < 0,2", "score ≥ 0,2"], yticklabels=["legítimo", "suspeito"],
           title="Matriz de confusão no corte do fast track")
    return fig, m


contrib = contribuicoes(modelo.booster_, Xte).drop(columns="base")
importancia = contrib.abs().mean().sort_values()


def _figura_importancia():
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.barh([ROTULOS[f] for f in importancia.index], importancia.to_numpy())
    ax.set(xlabel="média de |SHAP| (logito)", title="Importância das features (teste)")
    fig.tight_layout()
    return fig


def _figura_dispersao():
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for i, f in enumerate(importancia.index):
        valor = Xte[f].cat.codes.astype(float) if hasattr(Xte[f], "cat") else Xte[f].astype(float)
        cor = (valor - valor.min()) / (valor.max() - valor.min()) if valor.max() > valor.min() else valor * 0
        jitter = np.random.default_rng(i).uniform(-0.3, 0.3, len(valor))
        ax.scatter(contrib[f], i + jitter, c=cor.fillna(0.5), cmap="coolwarm", s=3, alpha=0.6)
    ax.set_yticks(range(len(importancia)), [ROTULOS[f] for f in importancia.index])
    ax.axvline(0, color="grey", lw=0.8)
    ax.set(xlabel="SHAP (logito): > 0 aumenta a suspeita", title="SHAP por sinistro (cor: valor da feature, azul = baixo)")
    fig.tight_layout()
    return fig


fig_conf, matriz = _figura_confusao(teste["suspeito"], score_te)
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_figure(_figura_roc(teste["suspeito"], score_te), "curva_roc.png")
    mlflow.log_figure(_figura_pr(teste["suspeito"], score_te), "curva_pr.png")
    mlflow.log_figure(fig_conf, "matriz_confusao_limiar_0_2.png")
    mlflow.log_figure(_figura_importancia(), "shap_importancia.png")
    mlflow.log_figure(_figura_dispersao(), "shap_dispersao.png")
    mlflow.log_table(contrib.sample(min(500, len(contrib)), random_state=0).round(4), "shap_valores_teste.json")
plt.close("all")
print("Matriz de confusão (linhas: legítimo, suspeito; colunas: score < 0,2, score ≥ 0,2):")
print(matriz)
display(pd.DataFrame({"feature": [ROTULOS[f] for f in importancia.index[::-1]],  # noqa: F821
                      "media_abs_shap": importancia.to_numpy()[::-1].round(4)}))

Matriz de confusão (linhas: legítimo, suspeito; colunas: score < 0,2, score ≥ 0,2):
[[5588   54]
 [  22  869]]


| feature | media_abs_shap |
|---|---|
| Desvio de preço | 0.8387 |
| Valor reclamado | 0.6737 |
| Evidência climática | 0.6063 |
| Valor ÷ LMI | 0.5997 |
| Participação do prestador (30 dias) | 0.5541 |
| Dias desde a contratação | 0.5197 |
| Laudo com dano anterior | 0.4008 |
| Causa declarada | 0.1953 |
| Cobertura válida | 0.104 |
| Ramo da apólice | 0.0087 |


## 4. Registro no Unity Catalog (*models from code*)

- O modelo servido recebe as colunas de entrada da silver, tipadas por `preparar_entrada`, e devolve
  `score_suspeita` e `fatores_suspeita`.
- O `features.py` do treino vai como artefato e é carregado por caminho, com nome de módulo único. Assim, o
  modelo usa o próprio código mesmo num processo que já importou `regras`.
- Antes de seguir, o modelo carregado do UC precisa reproduzir o score do booster.

In [ ]:
pasta = Path(tempfile.mkdtemp())
modelo.booster_.save_model(str(pasta / "booster.txt"))
(pasta / "config.json").write_text(json.dumps({"features": list(Xtr.columns), "com_clima_bruto": False}), encoding="utf-8")
with mlflow.start_run(run_id=RUN_ID):
    info = mlflow.pyfunc.log_model(
        name="modelo",
        python_model=str(Path(ml.__file__).with_name("modelo_pyfunc.py")),
        artifacts={"booster": str(pasta / "booster.txt"), "config": str(pasta / "config.json"),
                   "features": str(Path(regras.__file__).with_name("features.py"))},
        signature=assinatura(),
        input_example=preparar_entrada(teste.head(5)),
        pip_requirements=[f"mlflow=={VERSOES['mlflow']}", f"lightgbm=={VERSOES['lightgbm']}",
                          f"pandas=={VERSOES['pandas']}", f"numpy=={VERSOES['numpy']}"],
        registered_model_name=NOME_MODELO,
    )
VERSAO_MODELO = str(info.registered_model_version)
carregado = mlflow.pyfunc.load_model(f"models:/{NOME_MODELO}/{VERSAO_MODELO}")
saida_teste = carregado.predict(preparar_entrada(teste))
assert np.allclose(saida_teste["score_suspeita"].to_numpy(), score_te), "o modelo do UC não reproduz o score do treino"
print(f"registrado: {NOME_MODELO} versão {VERSAO_MODELO}; score do UC = score do booster em {len(teste)} sinistros")

🔗 View Logged Model at: <workspace>/ml/experiments/271986086899747/models/m-3f1928dc68af4404ba56445a35331fda
2026/11/21 03:07:49 INFO mlflow.pyfunc: Validating input example against model signature
Registered model 'fe_bar_sinistros_catalog.ml.modelo_suspeita' already exists. Creating a new version of this model...
🔗 Created version '3' of model 'fe_bar_sinistros_catalog.ml.modelo_suspeita': <workspace>/explore/data/models/fe_bar_sinistros_catalog/ml/modelo_suspeita/version/3


registrado: fe_bar_sinistros_catalog.ml.modelo_suspeita versão 3; score do UC = score do booster em 6533 sinistros


## 5. Checagem fora do tempo: lotes de validação do evento de 2026

- Estes avisos passaram pelo pipeline real: OCR dos PDFs, extração por IA e cruzamentos.
- O modelo usado é o **carregado do UC**.
- Só entram os lotes de validação (`lotes_validacao`); os demais são o teste final da fase 5 e não são tocados.
- O gabarito do evento serve só para medir; nunca é usado no treino.

In [ ]:
gab_ev = spark.read.csv(f"{GABARITO}/gabarito_evento_2026.csv", header=True).select("id_aviso", "padrao")  # noqa: F821
lote = (spark.table(f"{SILVER}.avisos")  # noqa: F821
        .select("id_aviso", F.regexp_extract("_arquivo", r"avisos_lote_([0-9]+)", 1).cast("int").alias("lote")))
evento = (spark.table(f"{SILVER}.sinistros_enriquecidos")  # noqa: F821
          .select("id_aviso", "data_aviso", *COLUNAS_ENTRADA, *COLUNAS_CLIMA_BRUTO)
          .join(lote, "id_aviso").where(F.col("lote").isin(LOTES_VALIDACAO))
          .join(gab_ev, "id_aviso", "left").orderBy("id_aviso").toPandas())
METRICAS_EVENTO, tabela_padrao_evento = {}, pd.DataFrame(columns=["padrao", "n"])
if evento.empty:
    print(f"Nenhum aviso dos lotes {LOTES_VALIDACAO} no pipeline ainda: checagem fora do tempo pulada.")
else:
    evento["suspeito"] = rotulo_suspeito(evento["padrao"])
    saida_ev = carregado.predict(preparar_entrada(evento))
    evento = evento.assign(score_suspeita=saida_ev["score_suspeita"].to_numpy(),
                           fatores_suspeita=saida_ev["fatores_suspeita"].to_numpy())
    regra_ev = regra_base(evento)
    METRICAS_EVENTO = (metricas(evento["suspeito"], evento["score_suspeita"], "evento_")
                       | metricas_fast_track(evento, evento["suspeito"], evento["score_suspeita"], "evento_"))
    METRICAS_REGRA_EVENTO = (metricas(evento["suspeito"], regra_ev, "regra_evento_")
                             | metricas_fast_track(evento, evento["suspeito"], regra_ev, "regra_evento_"))
    tabela_padrao_evento = por_padrao(evento["padrao"], evento["score_suspeita"]).merge(
        por_padrao(evento["padrao"], regra_ev)[["padrao", "pct_acima_limiar"]].rename(
            columns={"pct_acima_limiar": "pct_bloqueado_pela_regra"}), on="padrao")
    with mlflow.start_run(run_id=RUN_ID):
        mlflow.log_metrics(METRICAS_EVENTO | METRICAS_REGRA_EVENTO)
        mlflow.log_table(tabela_padrao_evento, "por_padrao_evento_validacao.json")
    display(pd.DataFrame([{"metrica": k.removeprefix("evento_"), "modelo": round(v, 4),  # noqa: F821
                           "regra_base": round(METRICAS_REGRA_EVENTO.get(f"regra_{k}", float("nan")), 4)}
                          for k, v in METRICAS_EVENTO.items()]))
    display(tabela_padrao_evento)  # noqa: F821
    exemplos = (evento.sort_values("score_suspeita", ascending=False).groupby("padrao").head(1)
                [["id_aviso", "padrao", "score_suspeita", "fatores_suspeita"]].round({"score_suspeita": 3}))
    display(exemplos)  # noqa: F821

| metrica | modelo | regra_base |
|---|---|---|
| auc | 0.9833 | 0.9704 |
| ap | 0.9706 | 0.9161 |
| precisao_top10 | 0.9875 | 0.95 |
| recall_top10 | 0.4247 | 0.4086 |
| recall_top10_teto | 0.4301 | 0.4301 |
| suspeitos_abaixo_limiar | 0.043 | 0.043 |
| legitimos_abaixo_limiar | 0.9886 | 0.9837 |
| n | 800.0 | 800.0 |
| n_suspeitos | 186.0 | 186.0 |
| ft_elegiveis | 403.0 | 403.0 |
| ft_suspeitos_elegiveis | 7.0 | 7.0 |
| ft_suspeitos_liberados | 3.0 | 3.0 |
| ft_legitimos_liberados_pct | 0.9975 | 0.9874 |


| padrao | n | score_mediano | pct_acima_limiar | pct_bloqueado_pela_regra |
|---|---|---|---|---|
| apolice_recente | 13 | 0.9947471672035431 | 100.0 | 100.0 |
| dano_preexistente | 9 | 0.994036346194341 | 100.0 | 100.0 |
| legitimo_complexo | 178 | 0.004730831241291707 | 2.8 | 2.2 |
| legitimo_simples | 436 | 0.001154209896494515 | 0.5 | 1.4 |
| orcamento_inflado | 25 | 0.9976394965862757 | 100.0 | 100.0 |
| prestador_concentrado | 5 | 0.001958158559669531 | 0.0 | 0.0 |
| sem_cobertura | 16 | 0.9992964448555535 | 100.0 | 100.0 |
| sem_evento_climatico | 118 | 0.9717905934586981 | 97.5 | 97.5 |


| id_aviso | padrao | score_suspeita | fatores_suspeita |
|---|---|---|---|
| AV-2026-000410 | sem_cobertura | 1.0 | Causa sem cobertura contratada |
| AV-2026-000774 | orcamento_inflado | 0.999 | Preço acima da tabela de referência |
| AV-2026-000635 | dano_preexistente | 0.998 | Laudo menciona dano anterior ao evento |
| AV-2026-000047 | apolice_recente | 0.998 | Apólice contratada há menos de 30 dias; Valor reclamado alto; Valor próximo do limite da cobertura (LMI) |
| AV-2026-000037 | sem_evento_climatico | 0.997 | Evidência climática contrária ao evento declarado; Apólice contratada há menos de 30 dias; Valor reclamado alto |
| AV-2026-000028 | legitimo_complexo | 0.994 | Apólice contratada há menos de 30 dias |
| AV-2026-000439 | legitimo_simples | 0.358 | Apólice contratada há menos de 30 dias |
| AV-2026-000748 | prestador_concentrado | 0.013 |  |


## 6. Ablação: e se o clima bruto entrasse no modelo?

A decisão foi tomada antes, pelo raciocínio de domínio (D-021). Esta ablação só mostra o efeito: é um run
aninhado, sem registro no UC. Ela treina com chuva, rajada e distância até a estação e mede no mesmo teste e nos
mesmos lotes de validação.

In [ ]:
Xtr_c, Xte_c = montar_features(treino, com_clima_bruto=True), montar_features(teste, com_clima_bruto=True)
with mlflow.start_run(run_id=RUN_ID):
    with mlflow.start_run(run_name="ablacao_com_clima_bruto", nested=True):
        mlflow.log_params({**PARAMS_LGBM, "com_clima_bruto": True, "features_treino_versao": VERSAO_FEATURES})
        mlflow.log_input(datasets["treino"], context="treino")
        ablacao = treinar(Xtr_c, treino["suspeito"])
        METRICAS_ABLACAO = metricas(teste["suspeito"], ablacao.predict_proba(Xte_c)[:, 1], "teste_")
        if not evento.empty:
            score_ablacao_ev = ablacao.predict_proba(montar_features(evento, com_clima_bruto=True))[:, 1]
            METRICAS_ABLACAO |= metricas(evento["suspeito"], score_ablacao_ev, "evento_")
            mlflow.log_table(por_padrao(evento["padrao"], score_ablacao_ev), "por_padrao_evento_validacao.json")
        mlflow.log_metrics(METRICAS_ABLACAO)
comparacao = pd.DataFrame([{"metrica": k, "modelo_registrado": round((METRICAS | METRICAS_EVENTO)[k], 4),
                            "ablacao_com_clima_bruto": round(v, 4)}
                           for k, v in METRICAS_ABLACAO.items() if not k.endswith(("_n", "_n_suspeitos"))])
display(comparacao)  # noqa: F821

2026/11/21 03:08:09 WARNING mlflow.data.spark_dataset: Encountered an unexpected exception while computing Spark dataset profile. Exception: [NOT_IMPLEMENTED] rdd is not implemented.


| metrica | modelo_registrado | ablacao_com_clima_bruto |
|---|---|---|
| teste_auc | 0.997 | 0.9973 |
| teste_ap | 0.9918 | 0.9924 |
| teste_precisao_top10 | 1.0 | 1.0 |
| teste_recall_top10 | 0.734 | 0.734 |
| teste_recall_top10_teto | 0.734 | 0.734 |
| teste_suspeitos_abaixo_limiar | 0.0247 | 0.0281 |
| teste_legitimos_abaixo_limiar | 0.9904 | 0.9918 |
| evento_auc | 0.9833 | 0.9878 |
| evento_ap | 0.9706 | 0.9811 |
| evento_precisao_top10 | 0.9875 | 1.0 |
| evento_recall_top10 | 0.4247 | 0.4301 |
| evento_recall_top10_teto | 0.4301 | 0.4301 |
| evento_suspeitos_abaixo_limiar | 0.043 | 0.043 |
| evento_legitimos_abaixo_limiar | 0.9886 | 0.9886 |


## 7. Portão de promoção (spec 6.1)

- **Piso absoluto:** AUC de teste ≥ 0,80 e no máximo 10% dos suspeitos do teste abaixo de 0,2. Sem ele, a versão
  nunca vira `champion`.
- **Comparação:** com o mesmo contrato de features, a versão nova só vira `champion` com AUC **maior** que o do
  `champion` atual no mesmo teste. Senão, vira `challenger`.
- **Outro contrato:** se a definição de alguma feature mudou, o `champion` atual receberia entradas com outro
  significado. O AUC dele não é comparável, e a versão nova assume se passar no piso (D-021).
- **Aliases:** são lidos da lista do modelo registrado, sem engolir erro de permissão ou rede. Ao promover, o
  `challenger` antigo é removido.

In [ ]:
client = MlflowClient(registry_uri="databricks-uc")
brutos = client.get_registered_model(NOME_MODELO).aliases or {}  # dict {alias: versão} (ou lista, em versões antigas)
aliases = {str(k): str(v) for k, v in (brutos.items() if isinstance(brutos, dict)
                                       else ((a.alias, a.version) for a in brutos))}
atual = client.get_model_version(NOME_MODELO, aliases["champion"]) if "champion" in aliases else None
contrato_igual = atual is None or (atual.tags or {}).get("contrato_features") == VERSAO_CONTRATO_FEATURES
auc_atual = None
if atual is not None and str(atual.version) != VERSAO_MODELO and contrato_igual:
    modelo_atual = mlflow.pyfunc.load_model(f"models:/{NOME_MODELO}/{atual.version}")
    auc_atual = metricas(teste["suspeito"], modelo_atual.predict(preparar_entrada(teste))["score_suspeita"])["auc"]
PASSOU_PISO = passa_piso(METRICAS)
ALIAS = decidir_promocao(METRICAS["teste_auc"], auc_atual, contrato_igual=contrato_igual, passou_piso=PASSOU_PISO)
client.set_registered_model_alias(NOME_MODELO, ALIAS, VERSAO_MODELO)
if ALIAS == "champion" and aliases.get("challenger") not in (None, VERSAO_MODELO):
    client.delete_registered_model_alias(NOME_MODELO, "challenger")  # candidata antiga deixa de valer
tags = {"teste_auc": f"{METRICAS['teste_auc']:.4f}", "corte_teste": CORTE_TESTE,
        "features_treino_versao": str(VERSAO_FEATURES), "alias_decidido": ALIAS, "passou_piso": str(PASSOU_PISO),
        "contrato_features": VERSAO_CONTRATO_FEATURES, "lotes_validacao": ",".join(map(str, LOTES_VALIDACAO)),
        "champion_anterior": "nenhum" if atual is None else f"v{atual.version}",
        "auc_champion_anterior": ("não comparável (outro contrato de features)" if not contrato_igual
                                  else "nenhum" if auc_atual is None else f"{auc_atual:.4f}")}
if METRICAS_EVENTO:
    tags["evento_validacao_auc"] = f"{METRICAS_EVENTO['evento_auc']:.4f}"
for chave, valor in tags.items():
    client.set_model_version_tag(NOME_MODELO, VERSAO_MODELO, chave, valor)
client.update_registered_model(NOME_MODELO, description=(
    "Score de suspeita de sinistros patrimoniais em eventos climáticos (LightGBM). Entrada: colunas de "
    "silver.sinistros_enriquecidos tipadas por regras.features.preparar_entrada; saída: score_suspeita (0 a 1) e "
    "fatores_suspeita. Score ≥ 0,2 bloqueia o fast track (spec 6.3). Alias champion = versão em uso; challenger = "
    "candidata que não superou o champion ou o piso."))
client.update_model_version(NOME_MODELO, VERSAO_MODELO, description=(
    f"Treino jan–set/2025 e teste out–dez/2025 (features_treino v{VERSAO_FEATURES}, contrato "
    f"{VERSAO_CONTRATO_FEATURES}). AUC teste {METRICAS['teste_auc']:.3f}"
    + (f"; AUC nos lotes de validação {LOTES_VALIDACAO} do evento {METRICAS_EVENTO['evento_auc']:.3f}"
       if METRICAS_EVENTO else "") + f". Run {RUN_ID}."))
print(f"versão {VERSAO_MODELO} → alias '{ALIAS}' (piso: {PASSOU_PISO}; champion anterior: "
      f"{None if atual is None else atual.version}; mesmo contrato: {contrato_igual}; AUC dele no mesmo teste: {auc_atual})")

versão 3 → alias 'champion' (piso: True; champion anterior: 2; mesmo contrato: False; AUC dele no mesmo teste: None)


In [ ]:
resultado = {
    "modelo": NOME_MODELO, "versao": VERSAO_MODELO, "alias": ALIAS, "run_id": RUN_ID,
    "features_treino_versao": VERSAO_FEATURES, "lotes_validacao": LOTES_VALIDACAO, "passou_piso": PASSOU_PISO,
    **{k: round(v, 4) for k, v in (METRICAS | METRICAS_REGRA | METRICAS_EVENTO).items()},
    **{f"ablacao_{k}": round(v, 4) for k, v in METRICAS_ABLACAO.items() if k.endswith(("auc", "abaixo_limiar"))},
}
print(json.dumps(resultado, ensure_ascii=False, indent=2))
dbutils.notebook.exit(json.dumps(resultado, ensure_ascii=False))  # noqa: F821

{"modelo": "fe_bar_sinistros_catalog.ml.modelo_suspeita", "versao": "3", "alias": "champion", "run_id": "95f3e85280404f49aff2e16a22878d63", "features_treino_versao": 5, "lotes_validacao": [1], "passou_piso": true, "treino_auc": 0.9997, "treino_ap": 0.9985, "treino_precisao_top10": 1.0, "treino_recall_top10": 0.6387, "treino_recall_top10_teto": 0.6387, "treino_suspeitos_abaixo_limiar": 0.0109, "treino_legitimos_abaixo_limiar": 0.9945, "treino_n": 13467.0, "treino_n_suspeitos": 2109.0, "teste_auc": 0.997, "teste_ap": 0.9918, "teste_precisao_top10": 1.0, "teste_recall_top10": 0.734, "teste_recall_top10_teto": 0.734, "teste_suspeitos_abaixo_limiar": 0.0247, "teste_legitimos_abaixo_limiar": 0.9904, "teste_n": 6533.0, "teste_n_suspeitos": 891.0, "teste_ft_elegiveis": 3028.0, "teste_ft_suspeitos_elegiveis": 94.0, "teste_ft_suspeitos_liberados": 14.0, "teste_ft_legitimos_liberados_pct": 0.9973, "regra_teste_auc": 0.96, "regra_teste_ap": 0.9233, "regra_teste_precisao_top10": 0.9893, "regra_test